In [7]:
!pip install ultralytics gradio opencv-python


In [8]:
from ultralytics import YOLO

# Load the trained model
model = YOLO("/content/best.pt")  # Update with your model path


In [9]:
import cv2
import numpy as np

def detect_image(image):
    # Convert Gradio image to OpenCV format
    image = np.array(image)

    # Run YOLO detection
    results = model(image)

    # Draw bounding boxes
    for result in results:
        for box in result.boxes.xyxy:
            x1, y1, x2, y2 = map(int, box[:4])
            label = int(box[5])  # Class label
            conf = box[4]  # Confidence score

            # Assign class names
            class_names = {0: "Accident", 2: "Car"}
            class_name = class_names.get(label, "Unknown")

            # Set color: Red for accident, Green for car
            color = (0, 0, 255) if label == 0 else (0, 255, 0)
            cv2.rectangle(image, (x1, y1), (x2, y2), color, 2)
            cv2.putText(image, f"{class_name} {conf:.2f}", (x1, y1 - 10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

    return image


In [10]:
import tempfile
import os

def detect_video(video_path):
    cap = cv2.VideoCapture(video_path)
    width = int(cap.get(3))
    height = int(cap.get(4))
    fps = int(cap.get(cv2.CAP_PROP_FPS))

    # Define output video file
    temp_file = tempfile.NamedTemporaryFile(delete=False, suffix=".mp4")
    output_path = temp_file.name
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        # Run YOLO detection
        results = model(frame)

        # Draw bounding boxes
        for result in results:
            for box in result.boxes.xyxy:
                x1, y1, x2, y2 = map(int, box[:4])
                label = int(box[5])  # Class label
                conf = box[4]  # Confidence score

                # Assign class names
                class_names = {0: "Accident", 2: "Car"}
                class_name = class_names.get(label, "Unknown")

                # Draw rectangle and label
                color = (0, 255, 0) if label == 2 else (0, 0, 255)  # Green for car, Red for accident
                cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
                cv2.putText(frame, f"{class_name} {conf:.2f}", (x1, y1 - 10),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

        out.write(frame)

    cap.release()
    out.release()

    return output_path


In [14]:
import gradio as gr

# Gradio interface for image detection
image_interface = gr.Interface(
    fn=detect_image,  # Single function for image processing
    inputs=gr.Image(type="numpy"),
    outputs=gr.Image(type="numpy"),
    title="YOLOv8 Car & Accident Detection - Image",
    description="Upload an image to detect cars and accidents."
)

# Gradio interface for video detection
video_interface = gr.Interface(
    fn=detect_video,  # Single function for video processing
    inputs=gr.Video(),
    outputs=gr.Video(),
    title="YOLOv8 Car & Accident Detection - Video",
    description="Upload a video to detect cars and accidents."
)

# Combine both interfaces using tabs
interface = gr.TabbedInterface([image_interface, video_interface], ["Image Detection", "Video Detection"])

# Launch the app
interface.launch()


Running Gradio in a Colab notebook requires sharing enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://30d78798e8be6a20ac.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
